In [25]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from itertools import combinations
from scipy.stats import ttest_ind
import pandas as pd

Entire the desired output_directory, model_name, and metric we care about. Then run all the code below

In [26]:
output_dir = 'models_failed'

output_dir = Path('/data/vision/polina/users/marcusbl/bin_class/outputs') / output_dir

model_name = 'model_loss'
metrics = ['auc', 'tpr_at_10%', 'tpr_at_20%', 'tpr_at_30%']

Parse results into Pandas DF

In [27]:
results = []

for test_dir in output_dir.iterdir():
    for run_dir in test_dir.iterdir():
        if not run_dir.is_dir():
            continue

        with open(run_dir / 'test_info' / 'metric_info.json') as f:
            metric_info = json.load(f)

        flat = {
            (model, metric): value
            for model, metrics in metric_info.items()
            for metric, value in metrics.items()
        }

        results.append({
            "test": test_dir.name,
            "run": run_dir.name,
            **flat
        })

df = pd.DataFrame(results).set_index(["test", "run"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df = df.sort_index(axis=1).sort_index()


In [28]:
df['model_auc']

acc    auc  epoch     f1  fn  fp    fpr   loss   prec  \
test      run                                                             
resnet101 run0  0.862  0.880     -1  0.681  47  41  0.082  0.372  0.696   
          run1  0.890  0.908     -1  0.670  52  20  0.038  0.370  0.785   
          run2  0.920  0.930     -1  0.667  21  32  0.055  0.213  0.624   
          run3  0.941  0.972     -1  0.785  16  23  0.040  0.140  0.755   
          run4  0.901  0.892     -1  0.640  58  13  0.022  0.311  0.829   
resnet152 run0  0.859  0.882     -1  0.648  58  32  0.064  0.331  0.722   
          run1  0.889  0.936     -1  0.695  42  31  0.058  0.242  0.728   
          run2  0.900  0.939     -1  0.633  17  49  0.084  0.230  0.538   
          run3  0.924  0.956     -1  0.734  18  32  0.056  0.187  0.683   
          run4  0.887  0.867     -1  0.571  67  14  0.024  0.305  0.794   
resnet18  run0  0.851  0.866     -1  0.678  41  54  0.109  0.390  0.649   
          run1  0.848  0.907     -1  0.650  32  68  0.128  0.328  0.578   
          run2  0.903  0.937     -1  0.628  20  44  0.075  0.213  0.551   
          run3  0.925  0.955     -1  0.741  17  32  0.056  0.199  0.686   
          run4  0.894  0.908     -1  0.675  42  34  0.057  0.268  0.699   
resnet34  run0  0.859  0.871     -1  0.694  39  51  0.103  0.386  0.667   
          run1  0.892  0.909     -1  0.708  39  32  0.060  0.288  0.729   
          run2  0.886  0.940     -1  0.611  15  60  0.102  0.240  0.496   
          run3  0.930  0.960     -1  0.758  15  31  0.054  0.195  0.699   
          run4  0.890  0.913     -1  0.672  40  39  0.066  0.310  0.675   
resnet50  run0  0.843  0.848     -1  0.587  70  30  0.060  0.385  0.703   
          run1  0.904  0.911     -1  0.707  49  14  0.026  0.372  0.844   
          run2  0.892  0.928     -1  0.603  20  51  0.087  0.235  0.514   
          run3  0.878  0.964     -1  0.672   5  75  0.132  0.297  0.522   
          run4  0.890  0.910     -1  0.695  31  48  0.081  0.286  0.652   

                recall   tn   tp    tpr  tpr_at_10%  tpr_at_20%  tpr_at_30%  
test      run                                                                
resnet101 run0   0.667  456   94  0.667       0.702       0.851       0.872  
          run1   0.584  511   73  0.584       0.736       0.840       0.888  
          run2   0.716  554   53  0.716       0.784       0.892       0.905  
          run3   0.816  547   71  0.816       0.897       0.966       1.000  
          run4   0.521  581   63  0.521       0.744       0.818       0.876  
resnet152 run0   0.589  465   83  0.589       0.709       0.823       0.879  
          run1   0.664  500   83  0.664       0.832       0.888       0.944  
          run2   0.770  537   57  0.770       0.770       0.905       0.946  
          run3   0.793  538   69  0.793       0.885       0.954       0.966  
          run4   0.446  580   54  0.446       0.702       0.785       0.835  
resnet18  run0   0.709  443  100  0.709       0.695       0.759       0.830  
          run1   0.744  463   93  0.744       0.728       0.816       0.888  
          run2   0.730  542   54  0.730       0.757       0.905       0.973  
          run3   0.805  538   70  0.805       0.851       0.920       0.977  
          run4   0.653  560   79  0.653       0.744       0.876       0.893  
resnet34  run0   0.723  446  102  0.723       0.716       0.801       0.851  
          run1   0.688  499   86  0.688       0.792       0.872       0.880  
          run2   0.797  526   59  0.797       0.797       0.905       0.959  
          run3   0.828  539   72  0.828       0.897       0.954       0.966  
          run4   0.669  555   81  0.669       0.760       0.893       0.926  
resnet50  run0   0.504  467   71  0.504       0.603       0.738       0.837  
          run1   0.608  517   76  0.608       0.776       0.840       0.912  
          run2   0.730  535   54  0.730       0.730       0.865       0.932  
          run3   0.943  495   82  0.943   

Average/Variance for desired metrics over all runs for the desired model

In [29]:
m = df[model_name]

stats = m.groupby(level='test').agg(['mean', 'var'])

stats = stats.sort_values(by=('auc', 'mean'), ascending=False)

stats[[(metric, stat) for metric in metrics for stat in ['mean', 'var']]]

auc           tpr_at_10%           tpr_at_20%            \
             mean       var       mean       var       mean       var   
test                                                                    
resnet34   0.9194  0.001076     0.7964  0.004468     0.8834  0.003177   
resnet152  0.9192  0.001798     0.7792  0.013705     0.8762  0.005285   
resnet18   0.9098  0.001927     0.7484  0.007541     0.8460  0.006256   
resnet101  0.9090  0.002295     0.7468  0.008908     0.8624  0.006587   
resnet50   0.9074  0.001024     0.7356  0.007429     0.8546  0.002644   

          tpr_at_30%            
                mean       var  
test                            
resnet34      0.9100  0.001860  
resnet152     0.9196  0.006083  
resnet18      0.9028  0.004867  
resnet101     0.9038  0.005925  
resnet50      0.9126  0.001609

Paired T-test for signficance. Note that it's paired b/c same seed for each 

In [31]:
from itertools import combinations
from scipy.stats import ttest_rel

rows = []

tests = m.index.get_level_values("test").unique()

for metric in metrics:
    for test_a, test_b in combinations(tests, 2):
        a = m.loc[test_a, metric].dropna()
        b = m.loc[test_b, metric].dropna()

        # align by run name so only matching seeds are compared
        paired = pd.concat(
            [a.rename("a"), b.rename("b")],
            axis=1,
            join="inner"
        ).dropna()

        if len(paired) < 2:
            t_stat = np.nan
            p_value = np.nan
        else:
            t_stat, p_value = ttest_rel(paired["a"], paired["b"])

        rows.append({
            "metric": metric,
            "test_a": test_a,
            "test_b": test_b,
            "mean_a": paired["a"].mean() if len(paired) else np.nan,
            "mean_b": paired["b"].mean() if len(paired) else np.nan,
            "diff": (paired["a"] - paired["b"]).mean() if len(paired) else np.nan,
            "n_pairs": len(paired),
            "t_stat": t_stat,
            "p_value": p_value,
        })

paired_tstats = (
    pd.DataFrame(rows)
    .set_index(["metric", "test_a", "test_b"])
    # .sort_values(["metric", "p_value"])
)

paired_tstats

mean_a  mean_b    diff  n_pairs    t_stat  \
metric     test_a    test_b                                                 
auc        resnet101 resnet152  0.9090  0.9192 -0.0102        5 -1.605951   
                     resnet18   0.9090  0.9098 -0.0008        5 -0.132344   
                     resnet34   0.9090  0.9194 -0.0104        5 -1.274755   
                     resnet50   0.9090  0.9074  0.0016        5  0.138498   
           resnet152 resnet18   0.9192  0.9098  0.0094        5  1.466960   
                     resnet34   0.9192  0.9194 -0.0002        5 -0.031031   
                     resnet50   0.9192  0.9074  0.0118        5  1.357668   
           resnet18  resnet34   0.9098  0.9194 -0.0096        5 -1.674702   
                     resnet50   0.9098  0.9074  0.0024        5  0.313625   
           resnet34  resnet50   0.9194  0.9074  0.0120        5  3.077935   
tpr_at_10% resnet101 resnet152  0.7468  0.7792 -0.0324        5 -1.424042   
                     resnet18   0.7468  0.7484 -0.0016        5 -0.123715   
                     resnet34   0.7468  0.7964 -0.0496        5 -2.669369   
                     resnet50   0.7468  0.7356  0.0112        5  0.502520   
           resnet152 resnet18   0.7792  0.7484  0.0308        5  1.667571   
                     resnet34   0.7792  0.7964 -0.0172        5 -0.632310   
                     resnet50   0.7792  0.7356  0.0436        5  1.144180   
           resnet18  resnet34   0.7484  0.7964 -0.0480        5 -2.422821   
                     resnet50   0.7484  0.7356  0.0128        5  0.602368   
           resnet34  resnet50   0.7964  0.7356  0.0608        5  1.912517   
tpr_at_20% resnet101 resnet152  0.8624  0.8762 -0.0138        5 -0.755623   
                     resnet18   0.8624  0.8460  0.0164        5  0.805958   
                     resnet34   0.8624  0.8834 -0.0210        5 -1.147524   
                     resnet50   0.8624  0.8546  0.0078        5  0.292679   
           resnet152 resnet18   0.8762  0.8460  0.0302        5  2.467136   
                     resnet34   0.8762  0.8834 -0.0072        5 -0.604338   
                     resnet50   0.8762  0.8546  0.0216        5  1.169946   
           resnet18  resnet34   0.8460  0.8834 -0.0374        5 -3.442074   
                     resnet50   0.8460  0.8546 -0.0086        5 -0.471305   
           resnet34  resnet50   0.8834  0.8546  0.0288        5  2.729398   
tpr_at_30% resnet101 resnet152  0.9038  0.9196 -0.0158        5 -0.941310   
                     resnet18   0.9038  0.9028  0.0010        5  0.055771   
                     resnet34   0.9038  0.9100 -0.0062        5 -0.276653   
                     resnet50   0.9038  0.9126 -0.0088        5 -0.354522   
           resnet152 resnet18   0.9196  0.9028  0.0168        5  1.482956   
                     resnet34   0.9196  0.9100  0.0096        5  0.431723   
                     resnet50   0.9196  0.9126  0.0070        5  0.326554   
           resnet18  resnet34   0.9028  0.9100 -0.0072        5 -0.501113   
                     resnet50   0.9028  0.9126 -0.0098        5 -0.675557   
           resnet34  resnet50   0.9100  0.9126 -0.0026        5 -0.373492   

                                 p_value  
metric     test_a    test_b               
auc        resnet101 resnet152  0.183558  
                     resnet18   0.901102  
                     resnet34   0.271411  
                     resnet50   0.896539  
           resnet152 resnet18   0.216283  
                     resnet34   0.976731  
                     resnet50   0.246111  
           resnet18  resnet34   0.169304  
                     resnet50   0.769480  
           resnet34  resnet50   0.037006  
tpr_at_10% resnet101 resnet152  0.227540  
                     resnet18   0.907508  
                     resnet34   0.055843  
                     resnet50   0.641707  
           resnet152 resnet18   0.170727  
                     resnet34   0.561524  
                     re